# 05 - Ponto de controle e CLP simulado

Fase 5: pessoa e pose, associações A e B, votação, decisão e escrita Modbus TCP no CLP
simulado, sem gravar imagens. O detalhamento está em `docs/fases/fase-5.md`.

Os parâmetros de `configs/ponto_de_controle.yaml` são provisórios (congelados na F6).
Para conferir caixas, regiões e registradores na tela, use `MOSTRAR = True` (`q` encerra);
só no computador local.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Guilh3rme/epi-sistema.git"

if "COLAB_RELEASE_TAG" in os.environ:
    raiz = Path("/content/epi-sistema")
    if not raiz.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(raiz)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))  # dispensa reiniciar o ambiente
else:
    raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

(raiz / "runs").mkdir(exist_ok=True)
os.chdir(raiz / "runs")  # pesos e resultados ficam em runs/, fora do git
print("raiz:", raiz.name)

In [ ]:
from epi.ambiente import configurar_ultralytics

configurar_ultralytics(raiz)  # antes de qualquer importação do ultralytics

import atexit
import time

from ultralytics.utils import ASSETS

from epi import clp, fluxo

MOSTRAR = False
config = fluxo.carregar_yaml("configs/ponto_de_controle.yaml")
config["mostrar"] = MOSTRAR
porta = config["clp"]["porta"]

## CLP simulado, em processo separado

In [ ]:
log_clp = raiz / "runs" / "clp_simulado.log"
with log_clp.open("w", encoding="utf-8") as destino:
    simulador = subprocess.Popen(
        [sys.executable, "-u", "-m", "epi.clp", "--porta", str(porta)],
        stdout=destino, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONPATH": str(raiz / "src"), "PYTHONIOENCODING": "utf-8"},
    )
atexit.register(simulador.kill)

limite = time.monotonic() + 60
while "CLP simulado em" not in log_clp.read_text(encoding="utf-8"):
    assert simulador.poll() is None and time.monotonic() < limite, log_clp.read_text(encoding="utf-8")
    time.sleep(0.2)
print(log_clp.read_text(encoding="utf-8").strip())

## Nível 2: pose real e detector roteirizado

A pessoa vem do `yolo26n.pt`; os EPIs são os do roteiro, fictícios. O vídeo é montado em
`runs/` a partir de `bus.jpg`.

In [ ]:
ROTEIRO = "configs/roteiro_demo.yaml"
IMAGENS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".gif", ".mp4", ".avi", ".mov", ".mkv"}


def midias():
    ignorar = {".venv", ".git", "data"}
    return {
        (str(p.relative_to(raiz)), p.stat().st_mtime_ns)
        for p in raiz.rglob("*")
        if p.suffix.lower() in IMAGENS and p.relative_to(raiz).parts[0] not in ignorar
    }


video = fluxo.montar_video(
    fluxo.carregar_yaml(ROTEIRO), ASSETS / "bus.jpg", raiz / "runs" / "f5" / "roteiro_demo.mp4"
)
demo = {**config, "modelos": {**config["modelos"], "roteiro": ROTEIRO}}
antes = midias()
resumo = fluxo.executar("runs/f5/roteiro_demo.mp4", demo)
assert resumo["falha"] is None, resumo["falha"]
assert midias() == antes, midias() ^ antes
print("nenhuma imagem ou vídeo gravado pelo fluxo")

In [ ]:
for d in resumo["decisoes"]:
    t = d["tempos"]
    clp_ms = d["escrita"]["latencia_ms"] if d["escrita"] and d["escrita"]["ok"] else None
    print(f"passagem {d['passagem']} {d['associacao']}{'*' if d['ativa'] else ' '} "
          f"{d['decisao']:<13} {d['motivo']:<16} ausentes={d['ausentes']} "
          f"quadros={t['quadros_desde_pessoa']} tempo={t['ate_decisao_s']:.2f}s"
          + (f" CLP={clp_ms:.2f}ms" if clp_ms is not None else ""))

obtido = {a: [d["decisao"] for d in resumo["decisoes"] if d["associacao"] == a] for a in "AB"}
assert obtido["A"] == ["LIBERADO", "BLOQUEADO", "INDETERMINADO", "LIBERADO"], obtido
assert obtido["B"] == ["LIBERADO", "BLOQUEADO", "INDETERMINADO", "BLOQUEADO"], obtido
assert resumo["falhas_clp"] == 0 and resumo["registradores_finais"][0] == 0

ambiente = resumo["ambiente"]
print()
print({k: ambiente[k] for k in ("gpu", "python", "torch", "ultralytics", "commit")})
for chave in ("quadros", "fps_video", "fps_laco", "fps_com_pessoa", "fluxo_acompanha_video",
              "custo_medio_ms_com_pessoa", "latencia_clp_ms", "registradores_finais", "eventos"):
    valor = resumo[chave]
    if chave == "eventos":
        valor = os.path.relpath(valor, raiz)
    print(f"{chave}: {valor}")
if not resumo["fluxo_acompanha_video"]:
    print("ATENÇÃO: com pessoa no quadro o fluxo foi mais lento que o vídeo; "
          "o tempo até a decisão não vale.")

## Vídeo de fumaça

Serve só para achar defeitos de funcionamento: não escolhe modelo, limiar, N ou região, e
nada dele é reportado. Sem um modelo da F3, o `yolo26n` não detecta EPIs.

In [ ]:
if config["video"] and (raiz / config["video"]).exists():
    fumaca = fluxo.executar(config["video"], config)
    print("falha:", fumaca["falha"], "| quadros:", fumaca["quadros"])
    for d in fumaca["decisoes"]:
        print(d["passagem"], d["associacao"], d["decisao"], d["motivo"])
else:
    print(f"AVISO: passo pulado; vídeo de fumaça não encontrado em {config['video']}.")

## Escritas recebidas pelo CLP e encerramento

In [ ]:
cliente = clp.cliente(porta)
registradores = clp.ler(cliente)
cliente.close()
simulador.kill()
simulador.wait()
print(log_clp.read_text(encoding="utf-8").strip())
print("registradores lidos ao final:", registradores)
assert registradores is not None and registradores[0] == 0
print("fase 5, nível 2 OK")